# Test: Qwen3-Embedding-0.6B (Lightweight)

Fast CPU embedding model (~1.2GB). Used when GPUs are occupied by LLM servers.
Embedding dim: 1024. Matryoshka supported (32-1024).

**Prerequisites:** Model files in `models/Qwen/Qwen3-Embedding-0.6B/`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from models.utils import Qwen3EmbeddingSmall

qwen_emb = Qwen3EmbeddingSmall()
print('Model config:')
qwen_emb.get_config()

## 1. Load Model (CPU)

In [ ]:
import time
t0 = time.time()
model = qwen_emb.load()
print(f'Loaded on: {qwen_emb.device} in {time.time()-t0:.1f}s')
print(f'Embedding dim: {qwen_emb.get_embedding_dimension()}')

## 2. Basic Encoding

In [ ]:
import numpy as np

# Single text
emb = qwen_emb.encode('Lateral movement detected from workstation WS-042')
print(f'Shape: {emb.shape}')
print(f'L2 norm: {np.linalg.norm(emb[0]):.4f}')

# Batch
texts = [
    'Multiple failed SSH login attempts from 10.0.5.12',
    'Unusual outbound DNS traffic to known C2 domain',
    'Ransomware encryption detected on file server FS-01',
]
embs = qwen_emb.encode(texts)
print(f'Batch shape: {embs.shape}')

## 3. Query vs Document Encoding

In [ ]:
q_emb = qwen_emb.encode_queries('What hosts show signs of lateral movement?')
d_emb = qwen_emb.encode_documents('WS-042 initiated PsExec connection to DC-01')
print(f'Query shape: {q_emb.shape}')
print(f'Doc shape: {d_emb.shape}')

# Asymmetric check
test = 'lateral movement attack'
eq = qwen_emb.encode_queries(test)
ed = qwen_emb.encode_documents(test)
sim = (eq @ ed.T).item()
print(f'Same text query vs doc: {sim:.4f} (should be <1.0 if asymmetric)')

## 4. Matryoshka Dimensions

In [ ]:
for dim in [1024, 512, 256, 64, 32]:
    qwen_emb.set_embedding_dim(dim)
    e = qwen_emb.encode('test')
    print(f'  dim={dim}: shape={e.shape}, norm={np.linalg.norm(e[0]):.4f}')
qwen_emb.set_embedding_dim(1024)  # reset

## 5. Similarity

In [ ]:
sim = qwen_emb.similarity(
    ['lateral movement', 'PsExec execution'],
    ['Attacker moved laterally via PsExec', 'Daily backup completed']
)
print('Similarity matrix:')
print(np.round(sim, 3))

## 6. Write-Boundary Filter Test (P4)

In [ ]:
evidence = 'Lateral movement detected from workstation to domain controller'
legit = 'APT41 exploiting CVE-2024-38812 deploying backdoor with WMI lateral movement'
junk = 'Chocolate cake recipe: flour, sugar, cocoa, eggs, buttermilk'

sim_legit = qwen_emb.similarity(legit, evidence)[0][0]
sim_junk = qwen_emb.similarity(junk, evidence)[0][0]

threshold = 0.5
print(f'Legit: {sim_legit:.4f} -> {"PASS" if sim_legit > threshold else "REJECT"}')
print(f'Junk:  {sim_junk:.4f} -> {"PASS" if sim_junk > threshold else "REJECT"}')
print(f'Filter works: {sim_legit > threshold and sim_junk < threshold}')

## 7. Document Ranking

In [ ]:
kb = [
    'CVE-2024-1234: Remote code execution in Apache Struts',
    'APT29 uses PsExec and WMI for lateral movement',
    'Best practices for firewall rule management',
    'MITRE ATT&CK T1570: Lateral Tool Transfer',
]
ranked = qwen_emb.rank_documents('How do attackers move laterally?', kb, top_k=3)
for r in ranked:
    print(f'  {r["score"]:.4f}: {r["text"][:60]}')

## 8. ChromaDB Integration

In [ ]:
from memory.embedding_adapter import ChromaEmbeddingAdapter
import chromadb

adapter = ChromaEmbeddingAdapter(
    encode_fn=qwen_emb.encode,
    query_fn=qwen_emb.encode_queries,
    adapter_name='qwen3_embedding_0.6b',
)

client = chromadb.Client()
coll = client.get_or_create_collection('test_small', embedding_function=adapter)
coll.add(documents=kb, ids=[f'd{i}' for i in range(len(kb))])
print(f'Collection: {coll.count()} docs')

results = coll.query(query_texts=['lateral movement'], n_results=2)
print('Query results:')
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f'  {dist:.4f}: {doc[:60]}')

client.delete_collection('test_small')
print('Cleanup done')

## 9. Speed Comparison vs 8B

In [ ]:
batch = ['Text number ' + str(i) for i in range(32)]

t0 = time.time()
_ = qwen_emb.encode(batch)
small_ms = (time.time() - t0) * 1000
print(f'0.6B (CPU): {small_ms:.0f}ms for 32 texts')
print(f'Estimated 8B (CPU): ~{small_ms*10:.0f}ms (10x slower)')
print(f'Estimated 8B (GPU): ~{small_ms*0.1:.0f}ms')

## 10. Config

In [ ]:
import json
print(json.dumps(qwen_emb.get_config(), indent=2))

## Summary

All tests passed if no cells raised exceptions.

| Feature | 0.6B | 8B |
|---------|------|----|
| Size | ~1.2GB | ~16GB |
| Dim | 1024 | 4096 |
| CPU load | ~2s | ~4s |
| CPU encode (32 texts) | ~2s | ~20s |
| Default for | ChromaDB + write filter | Experiments needing SOTA |